In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!git clone https://github.com/iesxz-c/Final.git
%cd Final

Cloning into 'Final'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 232 (delta 113), reused 182 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 206.83 KiB | 14.77 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/Final


In [4]:
!nvidia-smi


Thu Sep 17 06:48:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!git log -1 --oneline
!python --version
!python -c "import torch; print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"

1fd1421 (HEAD -> master, origin/master, origin/HEAD) Add API evaluation benchmark
Python 3.13.15
CUDA: True
GPU: Tesla T4


In [8]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 8.4 MB/s eta 0:00:00


In [9]:
!mkdir -p data
!cp /content/drive/MyDrive/data/phase2c_155.json data/phase2c_155.json

!python -c "import json; x=json.load(open('data/phase2c_155.json')); print('155 manifest:', len(x['videos']), 'videos')"

155 manifest: 155 videos


In [10]:
!find /content/drive/MyDrive/data/evidence -maxdepth 3 -type f \
  \( -name "ucf_events.json" -o -name "manifest.json" \) | sort

/content/drive/MyDrive/data/evidence/fused/manifest.json
/content/drive/MyDrive/data/evidence/phase2b/manifest.json
/content/drive/MyDrive/data/evidence/phase2c_ucf_large/ucf_events.json
/content/drive/MyDrive/data/evidence/phase2c_ucf/manifest.json
/content/drive/MyDrive/data/evidence/phase2c_ucf_remaining_702/manifest.json
/content/drive/MyDrive/data/evidence/phase2c_ucf_remaining_702/ucf_events.json
/content/drive/MyDrive/data/evidence/phase2c_ucf/ucf_events.json
/content/drive/MyDrive/data/evidence/phase2/manifest.json


In [11]:
!mkdir -p data/evidence/phase2c_ucf_large

!cp -r /content/drive/MyDrive/data/evidence/phase2c_ucf_large/* \
       data/evidence/phase2c_ucf_large/

In [13]:
import json

p = "data/evidence/phase2c_ucf_large/ucf_events.json"
x = json.load(open(p))

print("UCF events:", len(x))
print("Unique videos:", len({e["video_id"] for e in x}))

UCF events: 7927
Unique videos: 155


In [14]:
import os

p = "data/models/yolo11n.pt"
print("Exists:", os.path.exists(p))
if os.path.exists(p):
    print("Size:", round(os.path.getsize(p) / (1024**2), 2), "MB")

Exists: False


In [16]:
import os
os.makedirs("data/models", exist_ok=True)

from ultralytics import YOLO

model = YOLO("yolo11n.pt")
print("YOLO loaded successfully")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO loaded successfully


In [17]:
import os

p = "data/models/yolo11n.pt"

# If Ultralytics downloaded it to the repo root, move it to the project's expected path.
if os.path.exists("yolo11n.pt") and not os.path.exists(p):
    os.replace("yolo11n.pt", p)

print("Exists:", os.path.exists(p))
if os.path.exists(p):
    print("Size:", round(os.path.getsize(p) / (1024**2), 2), "MB")

Exists: True
Size: 5.35 MB


In [18]:
!python -m src.pipeline.extract_evidence \
  --subset data/phase2c_155.json \
  --output-dir data/evidence/phase2_155_smoke \
  --device cuda \
  --max-videos 1

usage: extract_evidence.py [-h] [--config CONFIG] [--subset SUBSET]
                           [--output-dir OUTPUT_DIR]
                           [--limit-videos LIMIT_VIDEOS]
                           [--max-frames MAX_FRAMES]
                           [--save-frames SAVE_FRAMES] [--device DEVICE]
                           [--skip-detection]
extract_evidence.py: error: unrecognized arguments: --max-videos 1


In [19]:
!python -m src.pipeline.extract_evidence \
  --subset data/phase2c_155.json \
  --output-dir data/evidence/phase2_155_smoke \
  --device cuda \
  --limit-videos 1


Detector: yolo11n on cuda
[1/1] anomaly/Abuse/Abuse028_x264.mp4
  48 frames @ 1.0fps (video 30.00fps), 33 detections

Wrote 1 videos, 48 observations, 33 detections -> data/evidence/phase2_155_smoke in 11.7s


In [20]:
!rm -rf data/evidence/phase2_155

!python -m src.pipeline.extract_evidence \
  --subset data/phase2c_155.json \
  --output-dir data/evidence/phase2_155 \
  --device cuda

Detector: yolo11n on cuda
[1/155] anomaly/Abuse/Abuse028_x264.mp4
  48 frames @ 1.0fps (video 30.00fps), 33 detections
[2/155] anomaly/Abuse/Abuse030_x264.mp4
  52 frames @ 1.0fps (video 30.00fps), 94 detections
[3/155] anomaly/Arrest/Arrest001_x264.mp4
  80 frames @ 1.0fps (video 30.00fps), 151 detections
[4/155] anomaly/Arrest/Arrest007_x264.mp4
  105 frames @ 1.0fps (video 30.00fps), 97 detections
[5/155] anomaly/Arrest/Arrest024_x264.mp4
  121 frames @ 1.0fps (video 30.00fps), 484 detections
[6/155] anomaly/Arrest/Arrest030_x264.mp4
  289 frames @ 1.0fps (video 30.00fps), 181 detections
[7/155] anomaly/Arrest/Arrest039_x264.mp4
  528 frames @ 1.0fps (video 30.00fps), 2457 detections
[8/155] anomaly/Arson/Arson007_x264.mp4
  209 frames @ 1.0fps (video 30.00fps), 7 detections
[9/155] anomaly/Arson/Arson009_x264.mp4
  25 frames @ 1.0fps (video 30.00fps), 7 detections
[10/155] anomaly/Arson/Arson010_x264.mp4
  106 frames @ 1.0fps (video 30.00fps), 258 detections
[11/155] anomaly/Arson/

In [21]:
import os

p = "/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1/Explosion/Explosion013_x264.mp4"

print("Exists:", os.path.exists(p))
if os.path.exists(p):
    print("Size:", round(os.path.getsize(p) / (1024**2), 2), "MB")

Exists: True
Size: 15.66 MB


In [22]:
import json

with open("data/phase2c_155.json") as f:
    manifest = json.load(f)

target = [
    v for v in manifest["videos"]
    if v["video_id"] == "anomaly/Explosion/Explosion013_x264.mp4"
]

print("Selected:", len(target))
print(target[0]["video_id"] if target else "NOT FOUND")

with open("data/explosion013_manifest.json", "w") as f:
    json.dump({"videos": target}, f, indent=2)

Selected: 1
anomaly/Explosion/Explosion013_x264.mp4


In [23]:
!rm -rf data/evidence/phase2_155_explosion013

!python -m src.pipeline.extract_evidence \
  --subset data/explosion013_manifest.json \
  --output-dir data/evidence/phase2_155_explosion013 \
  --device cuda

Detector: yolo11n on cuda
[1/1] anomaly/Explosion/Explosion013_x264.mp4
  111 frames @ 1.0fps (video 30.00fps), 62 detections

Wrote 1 videos, 111 observations, 62 detections -> data/evidence/phase2_155_explosion013 in 5.8s


In [24]:
import json, os, shutil

MAIN = "data/evidence/phase2_155"
FIX  = "data/evidence/phase2_155_explosion013"

print("MAIN:", os.listdir(MAIN))
print("FIX :", os.listdir(FIX))

MAIN: ['observations.json', 'videos.json', 'detections.json', 'manifest.json']
FIX : ['observations.json', 'videos.json', 'detections.json', 'manifest.json']


In [25]:
for root, dirs, files in os.walk(FIX):
    level = root.replace(FIX, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

phase2_155_explosion013/
  observations.json
  videos.json
  detections.json
  manifest.json


In [26]:
import json
import os
import shutil

MAIN = "data/evidence/phase2_155"
FIX  = "data/evidence/phase2_155_explosion013"

# Load all files
with open(f"{MAIN}/observations.json") as f:
    main_obs = json.load(f)
with open(f"{MAIN}/videos.json") as f:
    main_vids = json.load(f)
with open(f"{MAIN}/detections.json") as f:
    main_dets = json.load(f)
with open(f"{MAIN}/manifest.json") as f:
    main_manifest = json.load(f)

with open(f"{FIX}/observations.json") as f:
    fix_obs = json.load(f)
with open(f"{FIX}/videos.json") as f:
    fix_vids = json.load(f)
with open(f"{FIX}/detections.json") as f:
    fix_dets = json.load(f)
with open(f"{FIX}/manifest.json") as f:
    fix_manifest = json.load(f)

print("MAIN:")
print(" observations:", len(main_obs))
print(" videos:", len(main_vids))
print(" detections:", len(main_dets))

print("\nFIX:")
print(" observations:", len(fix_obs))
print(" videos:", len(fix_vids))
print(" detections:", len(fix_dets))

MAIN:
 observations: 16696
 videos: 155
 detections: 44318

FIX:
 observations: 111
 videos: 1
 detections: 62


In [27]:
# Inspect the Explosion013 entries and manifest structure

target = "anomaly/Explosion/Explosion013_x264.mp4"

print("=== MAIN VIDEO MATCH ===")
for v in main_vids:
    if v.get("video_id") == target:
        print(json.dumps(v, indent=2))

print("\n=== FIX VIDEO ===")
for v in fix_vids:
    print(json.dumps(v, indent=2))

print("\n=== MANIFEST KEYS ===")
print("MAIN:", main_manifest.keys())
print("FIX :", fix_manifest.keys())

print("\n=== MAIN MANIFEST ===")
print(json.dumps(main_manifest, indent=2)[:5000])

print("\n=== FIX MANIFEST ===")
print(json.dumps(fix_manifest, indent=2)[:5000])

=== MAIN VIDEO MATCH ===
{
  "video_id": "anomaly/Explosion/Explosion013_x264.mp4",
  "source_type": "anomaly",
  "ground_truth_category": "Explosion",
  "dataset_root": "/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1",
  "path": "Explosion/Explosion013_x264.mp4",
  "duration_seconds": 110.567,
  "fps": 30.0,
  "width": 320,
  "height": 240,
  "frame_count": 3317,
  "file_size_bytes": 16418748,
  "format": "mp4",
  "schema_version": "phase2a/v1"
}

=== FIX VIDEO ===
{
  "video_id": "anomaly/Explosion/Explosion013_x264.mp4",
  "source_type": "anomaly",
  "ground_truth_category": "Explosion",
  "dataset_root": "/content/drive/MyDrive/Anomaly-Videos-Part-1/Anomaly-Videos-Part-1",
  "path": "Explosion/Explosion013_x264.mp4",
  "duration_seconds": 110.567,
  "fps": 30.0,
  "width": 320,
  "height": 240,
  "frame_count": 3317,
  "file_size_bytes": 16418748,
  "format": "mp4",
  "schema_version": "phase2a/v1"
}

=== MANIFEST KEYS ===
MAIN: dict_keys(['schema_version', 'sub

In [28]:
import json
import os
import shutil
from datetime import datetime, timezone

MAIN = "data/evidence/phase2_155"
FIX  = "data/evidence/phase2_155_explosion013"

TARGET = "anomaly/Explosion/Explosion013_x264.mp4"

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
def load(name, root):
    with open(os.path.join(root, name), "r") as f:
        return json.load(f)

main_obs = load("observations.json", MAIN)
main_vids = load("videos.json", MAIN)
main_dets = load("detections.json", MAIN)
main_manifest = load("manifest.json", MAIN)

fix_obs = load("observations.json", FIX)
fix_vids = load("videos.json", FIX)
fix_dets = load("detections.json", FIX)
fix_manifest = load("manifest.json", FIX)

# ------------------------------------------------------------
# 2. Validate standalone result
# ------------------------------------------------------------
assert len(fix_vids) == 1
assert fix_vids[0]["video_id"] == TARGET
assert len(fix_obs) == 111
assert len(fix_dets) == 62
assert fix_manifest["videos_processed"] == 1
assert fix_manifest["videos_failed"] == 0

# ------------------------------------------------------------
# 3. Make a backup of the current production corpus
# ------------------------------------------------------------
BACKUP = MAIN + "_backup_before_explosion013_merge"

if os.path.exists(BACKUP):
    shutil.rmtree(BACKUP)

shutil.copytree(MAIN, BACKUP)

print("Backup created:", BACKUP)

# ------------------------------------------------------------
# 4. Remove any existing Explosion013 records from MAIN
# ------------------------------------------------------------

merged_obs = [
    x for x in main_obs
    if x.get("video_id") != TARGET
]

merged_dets = [
    x for x in main_dets
    if x.get("video_id") != TARGET
]

merged_vids = [
    x for x in main_vids
    if x.get("video_id") != TARGET
]

# Add the successful standalone result
merged_obs.extend(fix_obs)
merged_dets.extend(fix_dets)
merged_vids.extend(fix_vids)

# ------------------------------------------------------------
# 5. Deterministic ordering
# ------------------------------------------------------------
merged_vids.sort(key=lambda x: x["video_id"])

merged_obs.sort(
    key=lambda x: (
        x.get("video_id", ""),
        x.get("observation_id", ""),
        x.get("start_seconds", 0)
    )
)

merged_dets.sort(
    key=lambda x: (
        x.get("video_id", ""),
        x.get("observation_id", ""),
        x.get("detection_id", "")
    )
)

# ------------------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------------------

video_ids = [x["video_id"] for x in merged_vids]
assert len(video_ids) == len(set(video_ids)), "Duplicate video_id!"

obs_ids = [
    x["observation_id"]
    for x in merged_obs
    if "observation_id" in x
]
assert len(obs_ids) == len(set(obs_ids)), "Duplicate observation_id!"

det_ids = [
    x["detection_id"]
    for x in merged_dets
    if "detection_id" in x
]

if det_ids:
    assert len(det_ids) == len(set(det_ids)), "Duplicate detection_id!"

assert TARGET in video_ids, "Explosion013 missing from videos!"

target_obs = [
    x for x in merged_obs
    if x.get("video_id") == TARGET
]

target_dets = [
    x for x in merged_dets
    if x.get("video_id") == TARGET
]

assert len(target_obs) == 111
assert len(target_dets) == 62

# Every observation/detection must reference a known video
video_set = set(video_ids)

assert all(
    x.get("video_id") in video_set
    for x in merged_obs
)

assert all(
    x.get("video_id") in video_set
    for x in merged_dets
)

# ------------------------------------------------------------
# 7. Write merged production files
# ------------------------------------------------------------

with open(os.path.join(MAIN, "videos.json"), "w") as f:
    json.dump(merged_vids, f, indent=2)

with open(os.path.join(MAIN, "observations.json"), "w") as f:
    json.dump(merged_obs, f, indent=2)

with open(os.path.join(MAIN, "detections.json"), "w") as f:
    json.dump(merged_dets, f, indent=2)

# ------------------------------------------------------------
# 8. Update manifest
# ------------------------------------------------------------

manifest = dict(main_manifest)

manifest["videos_selected"] = 155
manifest["videos_processed"] = 155
manifest["videos_failed"] = 0
manifest["observations"] = len(merged_obs)
manifest["detections"] = len(merged_dets)
manifest["failures"] = []

# Keep original production subset/detector configuration.
# Update elapsed time to reflect the successful recovery.
manifest["elapsed_seconds"] = round(
    main_manifest.get("elapsed_seconds", 0)
    + fix_manifest.get("elapsed_seconds", 0),
    1
)

with open(os.path.join(MAIN, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 9. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PHASE 2A — 155 VIDEO CORPUS")
print("=" * 60)

print("Videos       :", len(merged_vids))
print("Observations :", len(merged_obs))
print("Detections   :", len(merged_dets))
print("Failures     :", len(manifest["failures"]))

print("\nExplosion013:")
print("  Observations:", len(target_obs))
print("  Detections :", len(target_dets))
print("  Present    :", TARGET in video_set)

print("\nIntegrity:")
print("  Duplicate videos       : PASS")
print("  Duplicate observations : PASS")
print("  Duplicate detections   : PASS" if det_ids else "  Detection IDs          : N/A")
print("  Unknown video refs     : PASS")

print("\nExpected:")
print("  Videos       = 155")
print("  Observations = 16807")
print("  Detections   = 44380")

print("\n" + "=" * 60)

if (
    len(merged_vids) == 155
    and len(merged_obs) == 16807
    and len(merged_dets) == 44380
    and len(manifest["failures"]) == 0
):
    print("✅ PHASE 2A 155 CORPUS: PASS")
else:
    print("❌ PHASE 2A 155 CORPUS: FAIL — DO NOT CONTINUE")

Backup created: data/evidence/phase2_155_backup_before_explosion013_merge

PHASE 2A — 155 VIDEO CORPUS
Videos       : 155
Observations : 16807
Detections   : 44380
Failures     : 0

Explosion013:
  Observations: 111
  Detections : 62
  Present    : True

Integrity:
  Duplicate videos       : PASS
  Duplicate observations : PASS
  Duplicate detections   : PASS
  Unknown video refs     : PASS

Expected:
  Videos       = 155
  Observations = 16807
  Detections   = 44380

✅ PHASE 2A 155 CORPUS: PASS


In [29]:
!python -m src.pipeline.extract_activity \
  --subset data/phase2c_155.json \
  --output-dir data/evidence/phase2b_155 \
  --device cuda

preprocessor_config.json: 100% 271/271 [00:00<00:00, 789kB/s]
config.json: 100% 22.9k/22.9k [00:00<00:00, 45.9MB/s]

model.safetensors: downloading bytes:  62% 216M/346M [00:02<00:01, 129MB/s, 19.0MB/s  ]
model.safetensors: reconstructing file:  74% 256M/346M [00:02<00:00, 92.3MB/s]
model.safetensors: downloading bytes: 100% 231M/231M [00:02<00:00, 79.5MB/s, 20.1MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:02<00:00, 119MB/s, 31.8MB/s  ] 
Activity model: MCG-NJU/videomae-base-finetuned-kinetics on cuda (16 frames @ 8.0fps, top-5)
[1/155] anomaly/Abuse/Abuse028_x264.mp4
  23 windows (video 30.00fps)
[2/155] anomaly/Abuse/Abuse030_x264.mp4
  25 windows (video 30.00fps)
[3/155] anomaly/Arrest/Arrest001_x264.mp4
  38 windows (video 30.00fps)
[4/155] anomaly/Arrest/Arrest007_x264.mp4
  50 windows (video 30.00fps)
[5/155] anomaly/Arrest/Arrest024_x264.mp4
  57 windows (video 30.00fps)
[6/155] anomaly/Arrest/Arrest030_x264.mp4
  136 windows (video 30.00fps)
[7/155] anomal

In [30]:
!python -m src.pipeline.fuse_evidence \
  --phase2 data/evidence/phase2_155 \
  --phase2b data/evidence/phase2b_155 \
  --phase2c data/evidence/phase2c_ucf_large \
  --subset data/phase2c_155.json \
  --output-dir data/evidence/fused_155


Fused 155 videos: 44380 detections + 7927 activities + 7927 events -> 7927 records, 155 incidents in 48.1s -> data/evidence/fused_155
Top video scores:
  anomaly/Explosion/Explosion021_x264.mp4: 0.9999 {'strength': 0.9998, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  anomaly/Explosion/Explosion033_x264.mp4: 0.9999 {'strength': 0.9997, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}
  anomaly/RoadAccidents/RoadAccidents002_x264.mp4: 0.9999 {'strength': 0.9998, 'persistence': 1.0, 'concentration': 1.0, 'agreement': 1.0}


In [31]:
import os
import shutil
from google.colab import drive

# ============================================================
# CONFIG
# ============================================================

drive.mount("/content/drive")

SRC_ROOT = "/content/Final"
DRIVE_ROOT = "/content/drive/MyDrive/data_155n"

os.makedirs(DRIVE_ROOT, exist_ok=True)

# ============================================================
# FOLDERS / FILES TO SAVE
# ============================================================

items = [
    "data/evidence/fused_155",
    "data/evidence/phase2_155",
    "data/evidence/phase2b_155",
    "data/evidence/phase2_155_explosion013",
    "data/evidence/phase2_155_backup_before_explosion013_merge",
    "data/evidence/phase2_155_smoke",
    "data/evidence/phase2c_ucf_large",
    "data/phase2c_155.json",
]

# ============================================================
# COPY
# ============================================================

print("=" * 70)
print("SAVING 155-VIDEO PROJECT DATA TO GOOGLE DRIVE")
print("=" * 70)

for item in items:
    src = os.path.join(SRC_ROOT, item)
    dst = os.path.join(DRIVE_ROOT, os.path.basename(item))

    if not os.path.exists(src):
        print(f"⚠️ SKIP — not found: {src}")
        continue

    print(f"\n📦 Copying: {item}")

    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)

        shutil.copytree(src, dst)

    else:
        shutil.copy2(src, dst)

    print(f"   → {dst}")

# ============================================================
# VERIFY
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING")
print("=" * 70)

for item in items:
    dst = os.path.join(DRIVE_ROOT, os.path.basename(item))

    if os.path.exists(dst):
        if os.path.isdir(dst):
            total_files = sum(
                len(files)
                for _, _, files in os.walk(dst)
            )
            print(f"✅ {os.path.basename(item)} — {total_files} files")
        else:
            size_mb = os.path.getsize(dst) / (1024 ** 2)
            print(f"✅ {os.path.basename(item)} — {size_mb:.2f} MB")
    else:
        print(f"❌ MISSING — {os.path.basename(item)}")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"\nDrive directory:")
print(DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVING 155-VIDEO PROJECT DATA TO GOOGLE DRIVE

📦 Copying: data/evidence/fused_155
   → /content/drive/MyDrive/data_155n/fused_155

📦 Copying: data/evidence/phase2_155
   → /content/drive/MyDrive/data_155n/phase2_155

📦 Copying: data/evidence/phase2b_155
   → /content/drive/MyDrive/data_155n/phase2b_155

📦 Copying: data/evidence/phase2_155_explosion013
   → /content/drive/MyDrive/data_155n/phase2_155_explosion013

📦 Copying: data/evidence/phase2_155_backup_before_explosion013_merge
   → /content/drive/MyDrive/data_155n/phase2_155_backup_before_explosion013_merge

📦 Copying: data/evidence/phase2_155_smoke
   → /content/drive/MyDrive/data_155n/phase2_155_smoke

📦 Copying: data/evidence/phase2c_ucf_large
   → /content/drive/MyDrive/data_155n/phase2c_ucf_large

📦 Copying: data/phase2c_155.json
   → /content/drive/MyDrive/data_155n/phase2c_155.json

VERIFYING
✅ fus